In [6226]:
import reframed
import pandas as pd
from pathlib import Path
import numpy as np
import sys

sys.path.append('../../code/7_GEM_reconstruction')

from ng_utils import *
from ng_tests import *
from model_qc_and_polish import curate_model, remove_duplicated_metabolites, remove_duplicated_reactions, test_balance


from collections import defaultdict


In [6227]:
repo_folder = Path("../..")
data_folder = repo_folder / "data" / "7_GEM_reconstruction"
gapfilled_folder = data_folder / 'models/gapfilled_FBA'
polished_folder = data_folder / 'models/polished'


In [6228]:
model_abbrvs = ['At']#, 'Ct', 'Ml', 'Oa']

In [6229]:
model_dict = {}
for abbrv in model_abbrvs:
    model_path = gapfilled_folder / f'{abbrv}_gapfilled.xml'
    model = reframed.load_cbmodel(str(model_path))
    model_dict[abbrv] = model

## Fix missing charges

In [6230]:
charge_fn = data_folder / 'missing_metabolite_charges_bigg.csv'
charge_dict = pd.read_csv(charge_fn, index_col=0)['charge'].to_dict()

In [6231]:
STILL_MISSING = []

In [6232]:
pd.isna(charge_dict['acysbmn'])

True

In [6233]:
fixed_charges = {
    'ACP': -1,
    'fad': -2,
    'fadh2': -2,
    'ddcaACP': -1,
    'hd2coa': -4,
    'ddecoa': -4,
    'focytc': 0,
    'ode2coa': -4,
    # 'hdd7coa': 2,
    # '3oodecoa': 2,
    # '3hodecoa':2,
    'dde2coa': -4,
    'ddecoa': -5,
    # '3htdccoa': -4,
    '3hdd5coa': -4,
    'hdd4_2_coa': -4,
    "lnlc2coa": -4,
    'dec4coa':-4,
    'ptal': 0,
    'dann': 2,
    'dtbt':0,
    'btn': 0,
    'fdxo_2_2': 6,
    'fdxrd': 4,
    'fdxox': 5,
    'malACP': -2,
    'dc2coa': -4,
    'nonacoa': -2,
    'nona': 1,
    'poctacoa': -4,
    'php2coa': -4,
    'R_3hphpa': 1,
    'phxa2coa': -4,
    'ppt2coa': -6,
    '4atb2coa': -2,
    'decoa': -3,
    'R_3hnonaa': -1,
    'pta': -1,
    'pdcacoa': -4,
    'pocta2coa': -4,
    'R_3hpocta':-1,
    'tdecoa': -4,
    'R_3hdd6e': -1,
    'R_3hppta':-3,
    'R_3hphpa':-1,
    'R_3hphxa': -1,
    'ssaltpp': -3,
    'hp2coa': -4,
    'R_3hcmrs7e': -2,
    'R_3hhpa':-1,
    'lnlccoa':-2,
    'lnlc':1,
    'fe3dcit':-3,
    'seramp':0,
    'tag6p__D':-2,
    'mi3p__D':-2,
    'm2bcoa':0,
    '3h4atb': 1,
    'fmn':-2,
    'fmnh2':-2,
    'ribflv': 0,
    'hmbpp': -4,
    '23dhbzs2':-1,
    '23dhbzs3': -1,
    'fe3dhbzs3': 3,
    'trnaglu': 0,
    'cinmcoa': -4,
    # 'cinnm':-4
    # 'dmlz': -1,
    # 'db4p': -2,
    # '4r5au':-2,
    'salchs4': -1,
    'feenter': 2,
    'salc': -1,
    'dmso2': 4,
    'phenona': -3,
    'phehpa': -1,
    'abg4': -3,
    'val__D': -2,
    # '2mecdp'


    

}

equal = {
    'M_dde2coa_c': ['M_R_3hcddec5ecoa_c', 'M_3hddccoa_c', 'M_3oddccoa_c', 'M_decoa_c'],
    'M_3hpnonacoa_c': ['M_pnona2coa_c', 'M_R_3hpnonacoa_c', 'M_3opnonacoa_c', 'M_phpcoa_c'],
    'M_ddecoa_c': ['M_3otdccoa_c', 'M_3htdccoa_c', 'M_tde2coa_c', 'M_R_3hcmrs7ecoa_c'], 
    'M_3hdd5coa_c': ['M_tded5_2_coa_c', 'M_tded5coa_c', 'M_3ohdd7coa_c', 'M_3hhdd7coa_c', 'M_hdd7_2_coa_c'],
    # 'M_td2coa_c': ['M_tde_2Z_coa_c'],
    "M_lnlc2coa_c": ["M_3hlnlccoa_c", 'M_3olnlccoa_c', 'M_hd_7_10_coa_c', 'M_hd710_2_coa_c', 'M_3hhd710coa_c',
                     'M_3ohd710coa_c', 'M_td_5_8_coa_c', 'M_td58_2_coa_c', 'M_R_3htd58coa_c'],
    'M_dc2coa_c': ['M_R3hdec4coa_c', 'M_dec4_2_coa_c', 'M_3hdcoa_c'],
    'M_dec4coa_c': ['M_3odd6coa_c', 'M_3hdd6coa_c', 'M_dd6_2_coa_c', 'M_dd_3_6_coa_c', 'M_R_3hdd6coa_c',
                    'M_dd_3_6_coa_c', 'M_3ohd58coa_c', 'M_3hhd58coa_c'],
    'M_pocta2coa_c': ['M_R_3hpoctacoa_c', 'M_3hpoctacoa_c', 'M_3opoctacoa_c', 'M_phxacoa_c'],
    'M_php2coa_c': ['M_3hphpcoa_c', 'M_R_3hphpcoa_c','M_3ophpcoa_c', 'M_pptcoa_c'],
    # 'M_pro__D_c':['M_1p2cbxl_c'],
    'M_6ath2coa_c': ['M_R_3h6athcoa_c', 'M_3h6athcoa_c'],
    'M_hptcoa_c' : ['M_3ononacoa_c', 'M_3hnonacoa_c', 'M_nona2coa_c', 'M_R_3hnonacoa_c'],
    'M_hp2coa_c': ['M_3hhpcoa_c', 'M_R_3hhpcoa_c', 'M_3ohpcoa_c', 'M_ptcoa_c', 
                    'M_pt2coa_c', 'M_3hptcoa_c', 'M_3optcoa_c', 'M_ppcoa_c'],
    'M_4atb2coa_c': ['M_3h4atbcoa_c'],
    'M_hd2coa_c': ['M_3hhdccoa_c', 'M_3ohdccoa_c'],
    'M_ode2coa_c': ['M_3hodecoa_c', 'M_3oodecoa_c', 'M_hdd7coa_c'],
    'M_hdd4_2_coa_c': ['M_3hhdd4coa_c', 'M_3ohdd4coa_c', 'M_tde_2Z_coa_c'],
    
    'M_poctacoa_c': ['M_3opdecacoa_c', 'M_3hpdecacoa_c', 'M_pdca2coa_c', 'M_td2coa_c'],
    # 'M_decoa_c': ['M_3oddccoa_c'],
    'M_phxa2coa_c': ['M_R_3hphxacoa_c', 'M_3hphxacoa_c', 'M_3ophxacoa_c', 'M_pbcoa_c'],
    'M_ppt2coa_c': ['M_R_3hpptcoa_c', 'M_3hpptcoa_c', 'M_3opptcoa_c', 'M_phppcoa_c'],
}

In [6234]:
reactions_to_delete = []

In [6235]:
def fix_charge(model):
    for short_id, charge in fixed_charges.items():
        for end in ['_c', '_p', '_e']:
            met_id = f'M_{short_id}{end}'
            try:
                met = model.metabolites[met_id]
                met.metadata['CHARGE'] = str(charge)
                # print(f'Setting fixed CHARGE for {met.id} in model {model.id} to {charge}')
            except KeyError:
                continue
        

    for m_id in model.metabolites:
        met = model.metabolites[m_id]
        try:
            charge = met.metadata['CHARGE']
        except KeyError:
            short_id = met.id[2:-2]
            charge = charge_dict.get(short_id, np.nan)
            if np.isfinite(charge):
                # print(f'Setting CHARGE for {met.id} in model {model.id} to {charge}')
                met.metadata['CHARGE'] = str(int(charge_dict[short_id]))
            else:
                print(f'No CHARGE for {short_id} in model {model.id} or in charge dict')
                STILL_MISSING.append(short_id)

    for key, equal_ids in equal.items():
        for met_id in equal_ids:
            try:
                met = model.metabolites.get(met_id, None)
                met.metadata['CHARGE'] = model.metabolites[key].metadata['CHARGE']
            except (KeyError, TypeError, AttributeError):
                print(f'No CHARGE for {key} in model {model.id} to set equal to {met_id}')


In [6236]:
len(set(STILL_MISSING))

0

# Save models

In [6237]:
for abbrv, model in model_dict.items():
    fix_charge(model)
    polished_path = polished_folder / f'{abbrv}.xml'
    # reframed.save_cbmodel(model, str(polished_path))

In [6238]:
at = model_dict['At']

In [6239]:
fix_charge(at)

In [6240]:
at_df = test_balance(at)

In [6241]:
len(at_df)

19

In [6242]:
j = 0
count_dict = defaultdict(int)
for i, row in at_df.iterrows():
    r_id = row['r_id']
    r = at.reactions[r_id]
    print(r, row['charge'])
    for met_id, coeff in r.stoichiometry.items():
        met = at.metabolites[met_id]
        count_dict[met_id] += 1
        print(f'  {met_id}: {coeff}, CHARGE={met.metadata.get("CHARGE", "NA")}')    
    j+=1
    # if j > 10:
    #     break

R_23CTI1: M_decoa_c --> M_dc2coa_c + M_h_c 1.0
  M_decoa_c: -1.0, CHARGE=-4
  M_dc2coa_c: 1.0, CHARGE=-4
  M_h_c: 1.0, CHARGE=1
R_ACOAD10f: M_fad_c + M_hptcoa_c + 2.0 M_h_c <-> M_fadh2_c + M_hp2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_hptcoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_hp2coa_c: 1.0, CHARGE=-4
R_ACOAD12f: M_fad_c + M_pdcacoa_c + 2.0 M_h_c <-> M_fadh2_c + M_pdca2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_pdcacoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_pdca2coa_c: 1.0, CHARGE=-4
R_ACOAD14f: M_fad_c + M_poctacoa_c + 2.0 M_h_c <-> M_fadh2_c + M_pocta2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_poctacoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_pocta2coa_c: 1.0, CHARGE=-4
R_ACOAD15f: M_fad_c + M_phpcoa_c + 2.0 M_h_c <-> M_fadh2_c + M_php2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_phpcoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_php2coa_c: 1.0, CHAR

In [6243]:
# Get sorted count dict
sorted_counts = sorted(count_dict.items(), key=lambda x: x[1], reverse=True)

In [6244]:
sorted_counts

[('M_h_c', 15),
 ('M_fad_c', 10),
 ('M_fadh2_c', 10),
 ('M_h2o_c', 3),
 ('M_atp_c', 2),
 ('M_adp_c', 2),
 ('M_pi_c', 2),
 ('M_fdxrd_c', 2),
 ('M_gln__L_c', 2),
 ('M_fdxo_2_2_c', 2),
 ('M_glu__L_c', 2),
 ('M_decoa_c', 1),
 ('M_dc2coa_c', 1),
 ('M_hptcoa_c', 1),
 ('M_hp2coa_c', 1),
 ('M_pdcacoa_c', 1),
 ('M_pdca2coa_c', 1),
 ('M_poctacoa_c', 1),
 ('M_pocta2coa_c', 1),
 ('M_phpcoa_c', 1),
 ('M_php2coa_c', 1),
 ('M_phxacoa_c', 1),
 ('M_phxa2coa_c', 1),
 ('M_pptcoa_c', 1),
 ('M_ppt2coa_c', 1),
 ('M_pbcoa_c', 1),
 ('M_pb2coa_c', 1),
 ('M_tdecoa_c', 1),
 ('M_tde2coa_c', 1),
 ('M_hdd4coa_c', 1),
 ('M_hdd4_2_coa_c', 1),
 ('M_dec4coa_c', 1),
 ('M_dec4_2_coa_c', 1),
 ('M_2dmmq8_c', 1),
 ('M_amet_c', 1),
 ('M_ahcys_c', 1),
 ('M_mql8_c', 1),
 ('M_fe3dhbzs3_p', 1),
 ('M_fe3dhbzs3_c', 1),
 ('M_akg_c', 1),
 ('M_2mecdp_c', 1),
 ('M_h2mb4p_c', 1),
 ('M_dmlz_c', 1),
 ('M_4r5au_c', 1),
 ('M_ribflv_c', 1),
 ('M_salchs4fe_p', 1),
 ('M_salchs4fe_c', 1),
 ('M_2kmb_c', 1),
 ('M_met__L_c', 1),
 ('M_cys__L_c', 1